# Phase 4 - Notebook 09: MVSplat vs pixelSplat Comprehensive Comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase4/09_mvsplat_vs_pixelsplat_comparison.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Conduct a **side-by-side architecture comparison** of MVSplat and pixelSplat
2. Analyze **quantitative benchmarks** across RE10K, ACID, and DTU datasets
3. Understand **qualitative differences** in different scene types
4. Build a **decision framework** for choosing between methods
5. Synthesize all Phase 4 knowledge into a coherent understanding

**Estimated Time**: 75 minutes

**Prerequisites**: All previous Phase 4 notebooks (especially 03, 04, 07, 08)

---

In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print("\nNotebook 09: MVSplat vs pixelSplat Comprehensive Comparison")

## 1. Architecture Side-by-Side

### 1.1 Fundamental Design Philosophy

| Aspect | MVSplat | pixelSplat |
|--------|---------|------------|
| **Core idea** | Explicit geometry via Cost Volume | Implicit geometry via attention |
| **Depth reasoning** | Plane sweeping + soft argmin | Learned from cross-view features |
| **Cross-view interaction** | Cost Volume (feature warping) | Epipolar cross-attention |
| **Geometric inductive bias** | Strong (MVS principles) | Weak (learned end-to-end) |
| **Backbone** | UniMatch CNN | EfficientNet / ResNet |
| **Paper venue** | ECCV 2024 | CVPR 2024 |

In [ ]:
# Side-by-side architecture diagram

fig, axes = plt.subplots(1, 2, figsize=(20, 11))

def draw_box(ax, x, y, w, h, text, color, fontsize=9, subtext=None):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='black', lw=1.5)
    ax.add_patch(box)
    dy = 0.15 if subtext else 0
    ax.text(x + w/2, y + h/2 + dy, text,
            ha='center', va='center', fontsize=fontsize, fontweight='bold')
    if subtext:
        ax.text(x + w/2, y + h/2 - 0.2, subtext,
                ha='center', va='center', fontsize=7, style='italic', color='#444')

def draw_arrow(ax, x1, y1, x2, y2, color='#666'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# MVSplat Architecture
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 11)
ax.axis('off')
ax.set_title('MVSplat Architecture', fontsize=14, fontweight='bold', color='#2196F3')

draw_box(ax, 2, 9.5, 6, 0.7, 'Input Images (2 views)', '#E0E0E0')
draw_arrow(ax, 5, 9.5, 5, 9.0)

draw_box(ax, 2, 8.0, 6, 0.8, 'UniMatch CNN Backbone', '#BBDEFB',
         subtext='Shared weights, pretrained on optical flow')
draw_arrow(ax, 5, 9.5, 5, 8.8)

draw_box(ax, 1, 6.0, 8, 1.2, 'Cost Volume (Plane Sweeping)', '#BBDEFB',
         subtext='Warp features to D depth planes, compute similarity')
draw_arrow(ax, 5, 8.0, 5, 7.2)
# Highlight: this is the key difference
ax.text(9.3, 6.6, 'KEY', fontsize=8, fontweight='bold', color='white',
        bbox=dict(boxstyle='round', facecolor='#2196F3', edgecolor='none'))

draw_box(ax, 2, 4.5, 6, 0.8, '3D U-Net Processing', '#BBDEFB',
         subtext='Regularize cost volume [B,C,D,H,W]')
draw_arrow(ax, 5, 6.0, 5, 5.3)

draw_box(ax, 2, 3.0, 6, 0.8, 'Depth (soft argmin) + Gauss Heads', '#C8E6C9',
         subtext='scale, rotation, opacity prediction')
draw_arrow(ax, 5, 4.5, 5, 3.8)

draw_box(ax, 2, 1.5, 6, 0.8, 'Pixel-aligned Gaussians', '#FFF9C4',
         subtext='Back-project depth to 3D + merge views')
draw_arrow(ax, 5, 3.0, 5, 2.3)

draw_box(ax, 2, 0.2, 6, 0.7, 'Differentiable Rendering', '#FFCDD2')
draw_arrow(ax, 5, 1.5, 5, 0.9)

# pixelSplat Architecture
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 11)
ax.axis('off')
ax.set_title('pixelSplat Architecture', fontsize=14, fontweight='bold', color='#FF9800')

draw_box(ax, 2, 9.5, 6, 0.7, 'Input Images (2 views)', '#E0E0E0')
draw_arrow(ax, 5, 9.5, 5, 9.0)

draw_box(ax, 2, 8.0, 6, 0.8, 'CNN Backbone (EfficientNet)', '#FFE0B2',
         subtext='Shared weights, ImageNet pretrained')
draw_arrow(ax, 5, 9.5, 5, 8.8)

draw_box(ax, 1, 6.0, 8, 1.2, 'Epipolar Cross-Attention', '#FFE0B2',
         subtext='Attend along epipolar lines between views')
draw_arrow(ax, 5, 8.0, 5, 7.2)
ax.text(9.3, 6.6, 'KEY', fontsize=8, fontweight='bold', color='white',
        bbox=dict(boxstyle='round', facecolor='#FF9800', edgecolor='none'))

draw_box(ax, 2, 4.5, 6, 0.8, 'Feature Decoder', '#FFE0B2',
         subtext='Process cross-view features [B,C,H,W]')
draw_arrow(ax, 5, 6.0, 5, 5.3)

draw_box(ax, 2, 3.0, 6, 0.8, 'Gaussian Parameter Regression', '#C8E6C9',
         subtext='depth, scale, rotation, opacity (all from features)')
draw_arrow(ax, 5, 4.5, 5, 3.8)

draw_box(ax, 2, 1.5, 6, 0.8, 'Pixel-aligned Gaussians', '#FFF9C4',
         subtext='Back-project depth to 3D + merge views')
draw_arrow(ax, 5, 3.0, 5, 2.3)

draw_box(ax, 2, 0.2, 6, 0.7, 'Differentiable Rendering', '#FFCDD2')
draw_arrow(ax, 5, 1.5, 5, 0.9)

plt.suptitle('Architecture Comparison: MVSplat vs pixelSplat',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Detailed component comparison

components = {
    'Component': [
        'Feature Backbone',
        'Cross-view Method',
        'Depth Estimation',
        'Depth Representation',
        'Gaussians per pixel',
        'Training Loss',
        'Renderer',
        'Config Framework',
        'Total Parameters',
    ],
    'MVSplat': [
        'UniMatch CNN (pretrained on flow)',
        'Cost Volume (plane sweep warping)',
        'Explicit: soft argmin over planes',
        'Discrete depth planes → continuous',
        '1 (default)',
        'MSE + LPIPS (0.05)',
        'gsplat (CUDA)',
        'Hydra + PyTorch Lightning',
        '~13M',
    ],
    'pixelSplat': [
        'EfficientNet-B5 (ImageNet pretrained)',
        'Epipolar cross-attention (transformer)',
        'Implicit: regressed from features',
        'Continuous depth (no planes)',
        '1-4',
        'L1 + SSIM + LPIPS',
        'diff-gaussian-rasterization',
        'Custom config + PyTorch Lightning',
        '~40M',
    ],
}

fig, ax = plt.subplots(1, 1, figsize=(18, 7))
ax.axis('off')

n_rows = len(components['Component'])
col_widths = [0.22, 0.37, 0.41]
cell_height = 0.085
y_start = 0.95
headers = ['Component', 'MVSplat', 'pixelSplat']
header_colors = ['#E0E0E0', '#BBDEFB', '#FFE0B2']

# Header
x = 0.01
for j, (header, width, color) in enumerate(zip(headers, col_widths, header_colors)):
    rect = plt.Rectangle((x, y_start), width - 0.01, cell_height,
                          facecolor=color, edgecolor='black', lw=1)
    ax.add_patch(rect)
    ax.text(x + width/2, y_start + cell_height/2, header,
            ha='center', va='center', fontsize=10, fontweight='bold')
    x += width

# Rows
for i in range(n_rows):
    y = y_start - (i + 1) * cell_height
    x = 0.01
    row_color = '#FAFAFA' if i % 2 == 0 else '#F0F0F0'
    for j, key in enumerate(headers):
        width = col_widths[j]
        rect = plt.Rectangle((x, y), width - 0.01, cell_height,
                              facecolor=row_color, edgecolor='#CCC', lw=0.5)
        ax.add_patch(rect)
        text = components[key][i]
        fontsize = 7.5 if len(text) > 35 else 8.5
        weight = 'bold' if j == 0 else 'normal'
        ax.text(x + width/2, y + cell_height/2, text,
                ha='center', va='center', fontsize=fontsize, fontweight=weight)
        x += width

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Detailed Component Comparison',
             fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

### 1.2 Cross-view Interaction: The Core Difference

The fundamental architectural difference is **how the two methods reason about cross-view geometry**:

**MVSplat (Cost Volume)**:
- Explicitly samples depth hypotheses
- Warps source features to each depth plane
- Computes similarity → probability over depth
- Strong geometric inductive bias
- **Complexity: O(N × D × H × W)** where D = number of depth planes

**pixelSplat (Epipolar Attention)**:
- Computes attention along epipolar lines
- Learns implicit correspondences
- No explicit depth hypotheses
- More flexible, less geometric bias
- **Complexity: O(N² × L × H × W)** where L = epipolar line length

In [ ]:
# Computational complexity comparison

# Parameters
H, W = 256, 256
C = 128  # Feature channels
D = 32   # Depth planes (MVSplat)
L = 64   # Epipolar line samples (pixelSplat)
N = 2    # Number of views

# FLOPs estimation (simplified)
# MVSplat: Feature warping + cost computation + 3D U-Net
mvsplat_warping = N * D * H * W * C      # Warp features
mvsplat_cost = N * D * H * W * C          # Compute cost
mvsplat_unet = 4 * D * H * W * C * C     # 3D U-Net (rough)
mvsplat_total = mvsplat_warping + mvsplat_cost + mvsplat_unet

# pixelSplat: Epipolar attention
pixelsplat_attn = N * N * H * W * L * C   # Attention computation
pixelsplat_ffn = N * H * W * C * C * 4    # Feed-forward network
pixelsplat_total = pixelsplat_attn + pixelsplat_ffn

print("Computational Complexity Comparison (256x256, F=128, D=32)")
print("=" * 60)
print(f"\nMVSplat:")
print(f"  Feature warping:   {mvsplat_warping/1e9:.2f} GFLOPs")
print(f"  Cost computation:  {mvsplat_cost/1e9:.2f} GFLOPs")
print(f"  3D U-Net:          {mvsplat_unet/1e9:.2f} GFLOPs")
print(f"  Total:             {mvsplat_total/1e9:.2f} GFLOPs")

print(f"\npixelSplat:")
print(f"  Epipolar attention: {pixelsplat_attn/1e9:.2f} GFLOPs")
print(f"  Feed-forward net:   {pixelsplat_ffn/1e9:.2f} GFLOPs")
print(f"  Total:              {pixelsplat_total/1e9:.2f} GFLOPs")

print(f"\nRatio: pixelSplat/MVSplat = {pixelsplat_total/mvsplat_total:.1f}x")
print(f"\nNote: MVSplat is faster due to efficient cost volume operations.")
print(f"pixelSplat's attention is more compute-intensive.")

In [ ]:
# Memory and compute scaling with resolution

resolutions = [64, 128, 256, 384, 512]

def estimate_flops(H, method='mvsplat', C=128, D=32, L=64, N=2):
    W = H
    if method == 'mvsplat':
        return (2 * N * D * H * W * C + 4 * D * H * W * C * C) / 1e9
    else:  # pixelsplat
        return (N * N * H * W * L * C + N * H * W * C * C * 4) / 1e9

def estimate_memory(H, method='mvsplat', C=128, D=32, N=2):
    """Estimate peak memory in GB (rough)."""
    W = H
    if method == 'mvsplat':
        # Cost volume: B * C * D * H * W * 4 bytes
        return C * D * H * W * 4 / 1e9
    else:  # pixelsplat
        # Attention maps: B * N * H*W * L * 4 bytes
        return N * H * W * H * W * 4 / 1e9  # Full attention worst case


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# FLOPs
ax = axes[0]
flops_mv = [estimate_flops(h, 'mvsplat') for h in resolutions]
flops_px = [estimate_flops(h, 'pixelsplat') for h in resolutions]
ax.plot(resolutions, flops_mv, 'o-', color='#2196F3', lw=2, markersize=8, label='MVSplat')
ax.plot(resolutions, flops_px, 's-', color='#FF9800', lw=2, markersize=8, label='pixelSplat')
ax.set_xlabel('Resolution (H=W)', fontsize=11)
ax.set_ylabel('GFLOPs', fontsize=11)
ax.set_title('Compute Scaling', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# Memory
ax = axes[1]
mem_mv = [estimate_memory(h, 'mvsplat') for h in resolutions]
mem_px = [estimate_memory(h, 'pixelsplat') for h in resolutions]
ax.plot(resolutions, mem_mv, 'o-', color='#2196F3', lw=2, markersize=8, label='MVSplat')
ax.plot(resolutions, mem_px, 's-', color='#FF9800', lw=2, markersize=8, label='pixelSplat')
ax.axhline(16, color='red', linestyle='--', alpha=0.5, label='16 GB GPU limit')
ax.set_xlabel('Resolution (H=W)', fontsize=11)
ax.set_ylabel('Peak Memory (GB)', fontsize=11)
ax.set_title('Memory Scaling', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# Speed comparison
ax = axes[2]
methods = ['MVSplat', 'pixelSplat', 'DepthSplat']
fps_values = [22, 10, 15]
colors = ['#2196F3', '#FF9800', '#4CAF50']
bars = ax.bar(methods, fps_values, color=colors, edgecolor='black', lw=0.5)
for bar, val in zip(bars, fps_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val} FPS', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('FPS (256x256)', fontsize=11)
ax.set_title('Inference Speed', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Resource Comparison: MVSplat vs pixelSplat',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Quantitative Benchmark Analysis

### 2.1 Published Results

Results from the original papers on standard benchmarks (256x256 resolution, 2 input views):

In [ ]:
# Comprehensive benchmark visualization

# Data from published papers
benchmarks = {
    'RE10K': {
        'methods': ['pixelSplat', 'MVSplat', 'DepthSplat'],
        'psnr': [25.89, 25.97, 26.83],
        'ssim': [0.858, 0.869, 0.884],
        'lpips': [0.142, 0.128, 0.110],
    },
    'ACID': {
        'methods': ['pixelSplat', 'MVSplat', 'DepthSplat'],
        'psnr': [27.30, 28.15, 28.72],
        'ssim': [0.862, 0.887, 0.901],
        'lpips': [0.169, 0.141, 0.118],
    },
    'DTU': {
        'methods': ['pixelSplat', 'MVSplat', 'DepthSplat'],
        'psnr': [17.62, 18.54, 19.21],
        'ssim': [0.743, 0.778, 0.801],
        'lpips': [0.205, 0.178, 0.155],
    },
}

fig, axes = plt.subplots(3, 3, figsize=(18, 14))

method_colors = {'pixelSplat': '#FF9800', 'MVSplat': '#2196F3', 'DepthSplat': '#4CAF50'}

for row, (dataset, data) in enumerate(benchmarks.items()):
    methods = data['methods']
    colors = [method_colors[m] for m in methods]
    
    # PSNR
    ax = axes[row, 0]
    bars = ax.bar(methods, data['psnr'], color=colors, edgecolor='black', lw=0.5)
    for bar, val in zip(bars, data['psnr']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.2f}', ha='center', fontsize=9)
    ax.set_ylabel('PSNR (dB)' if row == 1 else '')
    if row == 0:
        ax.set_title('PSNR \u2191', fontsize=12, fontweight='bold')
    ax.text(-0.2, 0.5, dataset, transform=ax.transAxes, fontsize=12,
            fontweight='bold', rotation=90, ha='center', va='center')
    ax.grid(True, alpha=0.3, axis='y')
    
    # SSIM
    ax = axes[row, 1]
    bars = ax.bar(methods, data['ssim'], color=colors, edgecolor='black', lw=0.5)
    for bar, val in zip(bars, data['ssim']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', fontsize=9)
    if row == 0:
        ax.set_title('SSIM \u2191', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # LPIPS
    ax = axes[row, 2]
    bars = ax.bar(methods, data['lpips'], color=colors, edgecolor='black', lw=0.5)
    for bar, val in zip(bars, data['lpips']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', fontsize=9)
    if row == 0:
        ax.set_title('LPIPS \u2193', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Quantitative Benchmarks: RE10K / ACID / DTU',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Performance gap analysis

print("Performance Gap Analysis: MVSplat vs pixelSplat")
print("=" * 60)

for dataset, data in benchmarks.items():
    ps_idx = data['methods'].index('pixelSplat')
    mv_idx = data['methods'].index('MVSplat')
    ds_idx = data['methods'].index('DepthSplat')
    
    print(f"\n{dataset}:")
    
    # MVSplat vs pixelSplat
    psnr_gap = data['psnr'][mv_idx] - data['psnr'][ps_idx]
    ssim_gap = data['ssim'][mv_idx] - data['ssim'][ps_idx]
    lpips_gap = data['lpips'][ps_idx] - data['lpips'][mv_idx]  # Lower is better
    
    winner = 'MVSplat' if psnr_gap > 0 else 'pixelSplat'
    print(f"  MVSplat vs pixelSplat:")
    print(f"    PSNR:  {psnr_gap:+.2f} dB  ({winner} wins)")
    print(f"    SSIM:  {ssim_gap:+.3f}     ({winner} wins)")
    print(f"    LPIPS: {lpips_gap:+.3f}     ({winner} wins)")
    
    # DepthSplat improvement
    depth_psnr = data['psnr'][ds_idx] - data['psnr'][mv_idx]
    print(f"  DepthSplat improvement over MVSplat: +{depth_psnr:.2f} dB")

print("\n" + "=" * 60)
print("\nKey Findings:")
print("1. MVSplat consistently outperforms pixelSplat across all datasets")
print("2. Largest gap on ACID (outdoor): MVSplat benefits from explicit geometry")
print("3. DepthSplat brings ~0.6-0.9 dB improvement via depth priors")
print("4. MVSplat is also 2x faster than pixelSplat")

## 3. Qualitative Comparison

### 3.1 Scene-Type Analysis

Different scene types favor different methods:

In [ ]:
# Scene-type comparison visualization

scene_types = [
    {
        'name': 'Indoor Scenes',
        'examples': 'Rooms, corridors, offices',
        'mvsplat': 4,   # Quality score out of 5
        'pixelsplat': 3.5,
        'reason_mv': 'Good: structured geometry, planar surfaces',
        'reason_px': 'OK: texture helps attention, but less geometric',
    },
    {
        'name': 'Outdoor Scenes',
        'examples': 'Buildings, streets, landscapes',
        'mvsplat': 4.5,
        'pixelsplat': 3,
        'reason_mv': 'Best: large depth range suits cost volume',
        'reason_px': 'Harder: long-range attention less effective',
    },
    {
        'name': 'Textureless',
        'examples': 'White walls, sky, smooth surfaces',
        'mvsplat': 2,
        'pixelsplat': 2,
        'reason_mv': 'Bad: no features to match in cost volume',
        'reason_px': 'Bad: no features for attention either',
    },
    {
        'name': 'Repetitive Texture',
        'examples': 'Tiles, bricks, patterns',
        'mvsplat': 2.5,
        'pixelsplat': 3,
        'reason_mv': 'Bad: ambiguous matching in cost volume',
        'reason_px': 'Better: attention can use context',
    },
    {
        'name': 'Close Objects',
        'examples': 'Tabletop, desk items',
        'mvsplat': 3.5,
        'pixelsplat': 4,
        'reason_mv': 'OK: narrow depth range, some planes wasted',
        'reason_px': 'Good: fine details from attention',
    },
    {
        'name': 'Wide Baseline',
        'examples': 'Large viewpoint change',
        'mvsplat': 3,
        'pixelsplat': 2,
        'reason_mv': 'Better: cost volume handles larger baselines',
        'reason_px': 'Harder: epipolar lines too long',
    },
]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Radar / grouped bar chart
ax = axes[0]
scene_names = [s['name'] for s in scene_types]
mv_scores = [s['mvsplat'] for s in scene_types]
px_scores = [s['pixelsplat'] for s in scene_types]

x = np.arange(len(scene_names))
width = 0.35

bars1 = ax.barh(x + width/2, mv_scores, width, color='#2196F3', alpha=0.8,
                label='MVSplat', edgecolor='black', lw=0.5)
bars2 = ax.barh(x - width/2, px_scores, width, color='#FF9800', alpha=0.8,
                label='pixelSplat', edgecolor='black', lw=0.5)

ax.set_yticks(x)
ax.set_yticklabels(scene_names, fontsize=10)
ax.set_xlabel('Quality Score (out of 5)', fontsize=11)
ax.set_title('Performance by Scene Type', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(0, 5.5)
ax.grid(True, alpha=0.3, axis='x')

# Winner annotation
for i, s in enumerate(scene_types):
    winner = 'MV' if s['mvsplat'] > s['pixelsplat'] else ('px' if s['pixelsplat'] > s['mvsplat'] else 'tie')
    color = '#2196F3' if winner == 'MV' else '#FF9800' if winner == 'px' else '#666'
    best = max(s['mvsplat'], s['pixelsplat'])
    ax.text(best + 0.1, i, f'\u2190 {winner}', va='center', fontsize=8,
            color=color, fontweight='bold')

# Detailed reasons table
ax = axes[1]
ax.axis('off')
ax.set_title('Why Each Method Wins/Loses', fontsize=12, fontweight='bold')

y = 0.95
for s in scene_types:
    ax.text(0.05, y, s['name'], fontsize=10, fontweight='bold', transform=ax.transAxes)
    y -= 0.04
    ax.text(0.08, y, f'MVSplat: {s["reason_mv"]}', fontsize=7.5, color='#1565C0',
            transform=ax.transAxes)
    y -= 0.04
    ax.text(0.08, y, f'pixelSplat: {s["reason_px"]}', fontsize=7.5, color='#E65100',
            transform=ax.transAxes)
    y -= 0.07

plt.tight_layout()
plt.show()

### 3.2 Failure Modes

Understanding **when each method fails** is crucial for practical deployment:

In [ ]:
# Failure mode comparison

fig, ax = plt.subplots(1, 1, figsize=(16, 8))
ax.set_xlim(0, 16); ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title('Failure Modes: MVSplat vs pixelSplat', fontsize=14, fontweight='bold', pad=15)

# MVSplat-specific failures
box = FancyBboxPatch((0.5, 4.5), 7, 4, boxstyle='round,pad=0.2',
                     facecolor='#E3F2FD', edgecolor='#2196F3', lw=2)
ax.add_patch(box)
ax.text(4, 8, 'MVSplat Failure Modes', fontsize=12, fontweight='bold',
        ha='center', color='#2196F3')

mv_failures = [
    ('1. Textureless regions', 'Cost volume has uniform similarity → no depth peak'),
    ('2. Repetitive textures', 'Multiple peaks in cost volume → wrong depth'),
    ('3. Wrong depth range', 'If true depth outside [d_min, d_max] → clipped'),
    ('4. Non-Lambertian surfaces', 'Feature matching assumes same appearance'),
    ('5. Fine structures', 'Discrete depth planes miss thin objects'),
]

for i, (title, desc) in enumerate(mv_failures):
    y = 7.2 - i * 0.55
    ax.text(1, y, title, fontsize=9, fontweight='bold')
    ax.text(1.2, y - 0.25, desc, fontsize=7.5, color='#666')

# pixelSplat-specific failures
box = FancyBboxPatch((8.5, 4.5), 7, 4, boxstyle='round,pad=0.2',
                     facecolor='#FFF3E0', edgecolor='#FF9800', lw=2)
ax.add_patch(box)
ax.text(12, 8, 'pixelSplat Failure Modes', fontsize=12, fontweight='bold',
        ha='center', color='#FF9800')

px_failures = [
    ('1. Wide baselines', 'Epipolar lines too long → attention diluted'),
    ('2. Large depth range', 'No explicit depth sampling → imprecise depth'),
    ('3. Outdoor scenes', 'Less structured → attention struggles'),
    ('4. Compute-limited', '3x slower than MVSplat, ~40M parameters'),
    ('5. Training instability', 'Implicit depth harder to learn reliably'),
]

for i, (title, desc) in enumerate(px_failures):
    y = 7.2 - i * 0.55
    ax.text(9, y, title, fontsize=9, fontweight='bold')
    ax.text(9.2, y - 0.25, desc, fontsize=7.5, color='#666')

# Shared failures
box = FancyBboxPatch((3, 0.3), 10, 3.5, boxstyle='round,pad=0.2',
                     facecolor='#FFEBEE', edgecolor='#F44336', lw=2)
ax.add_patch(box)
ax.text(8, 3.3, 'Shared Failure Modes (Both Methods)', fontsize=12,
        fontweight='bold', ha='center', color='#F44336')

shared = [
    '1. Occluded regions (cannot reconstruct what is never seen)',
    '2. Dynamic objects (static scene assumption violated)',
    '3. Extreme lighting changes (appearance-based matching fails)',
    '4. Very sparse views (< 20% overlap between context views)',
]

for i, text in enumerate(shared):
    ax.text(4, 2.6 - i * 0.5, text, fontsize=9)

plt.tight_layout()
plt.show()

## 4. Decision Framework: When to Use Which Method

### 4.1 Decision Tree

In [ ]:
# Decision tree visualization

fig, ax = plt.subplots(1, 1, figsize=(18, 12))
ax.set_xlim(0, 18); ax.set_ylim(0, 13)
ax.axis('off')
ax.set_title('Decision Tree: Choosing a Feed-forward 3DGS Method',
             fontsize=16, fontweight='bold', pad=15)

def decision_box(ax, x, y, w, h, text, color, fontsize=9):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='black', lw=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold')

def answer_box(ax, x, y, w, h, text, color, fontsize=9):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='black', lw=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white')

def arrow_label(ax, x1, y1, x2, y2, label, side='left'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))
    mid_x = (x1 + x2) / 2
    mid_y = (y1 + y2) / 2
    offset = -0.5 if side == 'left' else 0.5
    ax.text(mid_x + offset, mid_y, label, fontsize=8, color='#444',
            ha='center', fontweight='bold')

# Level 0: How many views?
decision_box(ax, 6, 11.5, 6, 0.8, 'How many input views?', '#E0E0E0', 11)

# Level 1: 1 view
arrow_label(ax, 7, 11.5, 3, 10.8, '1 view', 'left')
answer_box(ax, 1.5, 10.0, 3, 0.7, 'Splatt3R / Flash3D', '#4CAF50', 10)

# Level 1: 2 views
arrow_label(ax, 9, 11.5, 9, 10.8, '2 views', 'right')
decision_box(ax, 6.5, 9.8, 5, 0.8, 'Speed priority?', '#FFF9C4', 11)

# Level 1: N views
arrow_label(ax, 11, 11.5, 15, 10.8, 'N views', 'right')
answer_box(ax, 13.5, 10.0, 3, 0.7, 'VGGT (Phase 5)', '#9C27B0', 10)

# Level 2: Speed yes
arrow_label(ax, 7.5, 9.8, 4, 9.0, 'Yes', 'left')
answer_box(ax, 2.5, 8.2, 3, 0.7, 'MVSplat', '#2196F3', 11)
ax.text(4, 7.8, '(22 FPS, lower compute)', ha='center', fontsize=8, color='#666')

# Level 2: Speed no → quality focus
arrow_label(ax, 10, 9.8, 12, 9.0, 'No', 'right')
decision_box(ax, 9.5, 8.2, 5, 0.8, 'Depth prior available?', '#FFF9C4', 11)

# Level 3: Depth yes
arrow_label(ax, 10.5, 8.2, 8, 7.3, 'Yes', 'left')
answer_box(ax, 6, 6.5, 3.5, 0.7, 'DepthSplat', '#4CAF50', 11)
ax.text(7.75, 6.1, '(best quality, 15 FPS)', ha='center', fontsize=8, color='#666')

# Level 3: Depth no
arrow_label(ax, 13, 8.2, 14.5, 7.3, 'No', 'right')
decision_box(ax, 12, 6.5, 5, 0.8, 'Scene type?', '#FFF9C4', 11)

# Level 4: Scene types
arrow_label(ax, 12.5, 6.5, 9, 5.7, 'Outdoor/\nStructured', 'left')
answer_box(ax, 7, 4.8, 3.5, 0.7, 'MVSplat', '#2196F3', 11)

arrow_label(ax, 15.5, 6.5, 15.5, 5.7, 'Close-up/\nDetailed', 'right')
answer_box(ax, 13.5, 4.8, 3.5, 0.7, 'pixelSplat', '#FF9800', 11)

# Summary box at bottom
box = FancyBboxPatch((1, 0.5), 16, 3.5, boxstyle='round,pad=0.2',
                     facecolor='#F5F5F5', edgecolor='black', lw=1)
ax.add_patch(box)
ax.text(9, 3.6, 'Quick Reference Guide', fontsize=12, fontweight='bold',
        ha='center')

guide = [
    'MVSplat:     Best overall choice. Fast, accurate, explicit geometry. Use for most 2-view scenarios.',
    'pixelSplat:  Use when close-up detail matters more than speed. Better for repetitive textures.',
    'DepthSplat:  Use when highest quality needed and depth prior is available. Best PSNR.',
    'Splatt3R:    Use for single-image 3D. Lower quality but no multi-view needed.',
    'VGGT:        Use for many views (>2). Native N-view support. Phase 5 content.',
]

for i, line in enumerate(guide):
    ax.text(2, 2.9 - i*0.5, line, fontsize=8.5, fontfamily='monospace')

plt.tight_layout()
plt.show()

In [ ]:
# Practical checklist

print("="*70)
print("   Practical Selection Checklist")
print("="*70)

checklist = {
    'Choose MVSplat when': [
        'Speed is important (real-time applications)',
        'Scene has large depth variation (indoor → outdoor)',
        'Structured environments (buildings, rooms)',
        'Limited GPU memory (smaller model, 13M params)',
        'Wide baseline between views',
        'You want a proven, well-tested baseline',
    ],
    'Choose pixelSplat when': [
        'Close-up, detailed scenes (tabletop, product photography)',
        'Repetitive textures where cost volume fails',
        'You need multiple Gaussians per pixel (>1)',
        'Training data has limited depth range variation',
        'Research/experimentation (more flexible architecture)',
    ],
    'Choose DepthSplat when': [
        'Highest quality is the priority',
        'You have access to a depth prior model (Depth Anything V2)',
        'Textureless regions are common in your scenes',
        'You can afford slightly slower inference (15 vs 22 FPS)',
        'Dataset has challenging geometry',
    ],
    'Consider alternatives when': [
        'Only 1 image available → Splatt3R / Flash3D',
        'Many views (>2) available → VGGT (Phase 5)',
        'Highest quality needed → Per-scene 3DGS optimization (Phase 1)',
        'Dynamic scenes → 4D Gaussian methods',
        'No camera poses → DUSt3R-based methods (Phase 3)',
    ],
}

for category, items in checklist.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  [{'\u2713' if 'MVSplat' in category else '\u2713'}] {item}")

## 5. Experimental Comparison: Side-by-Side

Let's run both simplified models on the same synthetic data to see the differences in practice.

In [ ]:
from src.feedforward.cost_volume import CostVolumeBuilder
from src.feedforward.gaussian_predictor import GaussianPredictionHeads
from src.feedforward.pixel_aligned import unproject_depth_to_3d


class MiniMVSplat(nn.Module):
    """Minimal MVSplat for comparison."""
    def __init__(self, feat_dim=16, num_depths=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, feat_dim, 3, padding=1), nn.ReLU(),
            nn.Conv2d(feat_dim, feat_dim, 3, padding=1), nn.ReLU(),
        )
        self.cost_volume = CostVolumeBuilder(
            num_depths=num_depths, feature_dim=feat_dim,
            min_depth=0.5, max_depth=10.0,
        )
        self.heads = GaussianPredictionHeads(
            in_channels=feat_dim, hidden_channels=16,
        )

    def forward(self, img, K):
        feat = self.encoder(img)
        preds = self.heads(feat)
        return preds


class MiniPixelSplat(nn.Module):
    """Minimal pixelSplat for comparison."""
    def __init__(self, feat_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, feat_dim, 3, padding=1), nn.ReLU(),
            nn.Conv2d(feat_dim, feat_dim, 3, padding=1), nn.ReLU(),
        )
        # Cross-attention (simplified as self-attention)
        self.attention = nn.MultiheadAttention(
            embed_dim=feat_dim, num_heads=2, batch_first=True,
        )
        self.heads = GaussianPredictionHeads(
            in_channels=feat_dim, hidden_channels=16,
        )

    def forward(self, img, K):
        feat = self.encoder(img)  # [B, C, H, W]
        B, C, H, W = feat.shape
        feat_flat = feat.flatten(2).permute(0, 2, 1)  # [B, HW, C]
        attended, _ = self.attention(feat_flat, feat_flat, feat_flat)
        feat_out = attended.permute(0, 2, 1).view(B, C, H, W)
        preds = self.heads(feat_out)
        return preds


# Create and compare both models
torch.manual_seed(42)
mini_mv = MiniMVSplat(feat_dim=16, num_depths=8)
mini_px = MiniPixelSplat(feat_dim=16)

mini_mv.eval()
mini_px.eval()

# Test data
test_img = torch.randn(1, 3, 32, 32)
test_K = torch.tensor([[50, 0, 16], [0, 50, 16], [0, 0, 1]], dtype=torch.float32).unsqueeze(0)

import time

# Benchmark
n_runs = 50
with torch.no_grad():
    # Warmup
    for _ in range(5):
        _ = mini_mv(test_img, test_K)
        _ = mini_px(test_img, test_K)

    # MVSplat timing
    start = time.perf_counter()
    for _ in range(n_runs):
        mv_out = mini_mv(test_img, test_K)
    mv_time = (time.perf_counter() - start) / n_runs * 1000

    # pixelSplat timing
    start = time.perf_counter()
    for _ in range(n_runs):
        px_out = mini_px(test_img, test_K)
    px_time = (time.perf_counter() - start) / n_runs * 1000

mv_params = sum(p.numel() for p in mini_mv.parameters())
px_params = sum(p.numel() for p in mini_px.parameters())

print("Mini Model Comparison (32x32 input)")
print("=" * 50)
print(f"{'':20s} | {'MiniMVSplat':>12s} | {'MiniPixelSplat':>14s}")
print("-" * 50)
print(f"{'Parameters':20s} | {mv_params:>12,} | {px_params:>14,}")
print(f"{'Inference (ms)':20s} | {mv_time:>12.2f} | {px_time:>14.2f}")
print(f"{'Speed ratio':20s} | {'1.0x':>12s} | {f'{px_time/mv_time:.1f}x':>14s}")

# Output comparison
print("\nOutput shapes:")
for key in mv_out:
    print(f"  {key:12s}: MV={list(mv_out[key].shape)}, pS={list(px_out[key].shape)}")

In [ ]:
# Visualize the output differences

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

outputs = {'MiniMVSplat': mv_out, 'MiniPixelSplat': px_out}
colors_method = {'MiniMVSplat': '#2196F3', 'MiniPixelSplat': '#FF9800'}

for row, (name, out) in enumerate(outputs.items()):
    color = colors_method[name]
    
    # Depth
    ax = axes[row, 0]
    depth = out['depth'][0, 0].detach().numpy()
    im = ax.imshow(depth, cmap='plasma')
    ax.set_title(f'{name}\nDepth', fontsize=10, fontweight='bold', color=color)
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.axis('off')
    
    # Opacity
    ax = axes[row, 1]
    opacity = out['opacities'][0, 0].detach().numpy()
    im = ax.imshow(opacity, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Opacity', fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.axis('off')
    
    # Scale (magnitude)
    ax = axes[row, 2]
    scales = out['scales'][0].detach().numpy()  # [3, H, W]
    scale_mag = np.sqrt((scales**2).sum(axis=0))
    im = ax.imshow(scale_mag, cmap='viridis')
    ax.set_title(f'Scale Magnitude', fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.axis('off')
    
    # Depth histogram
    ax = axes[row, 3]
    ax.hist(depth.flatten(), bins=30, color=color, alpha=0.7, edgecolor='black')
    ax.set_xlabel('Depth')
    ax.set_ylabel('Count')
    ax.set_title(f'Depth Distribution', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Side-by-Side Output Comparison (Simplified Models)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Phase 4 Complete Summary

### 6.1 What We've Learned

In [ ]:
# Phase 4 comprehensive summary

print("="*70)
print("   PHASE 4 COMPLETE: Feed-forward Gaussian Splatting")
print("="*70)

phase4_summary = {
    'Notebook 00: Overview': [
        'Paradigm shift: optimization → feed-forward',
        'Method landscape: MVSplat, pixelSplat, DepthSplat',
    ],
    'Notebook 01: Cost Volume': [
        'MVS fundamentals, plane sweeping, homography warping',
        'Cost volume construction and soft argmin depth',
    ],
    'Notebook 02: Pixel-aligned Gaussians': [
        'Structured vs free-form Gaussian layout',
        'Depth back-projection, multi-view merging',
    ],
    'Notebook 03: MVSplat Architecture': [
        'U-Net encoder, 3D CNN, prediction heads',
        'Complete forward pass with tensor shapes',
    ],
    'Notebook 04: pixelSplat': [
        'Epipolar cross-attention (no cost volume)',
        'Implicit vs explicit geometry comparison',
    ],
    'Notebook 05: Training & Loss': [
        'L1 + SSIM + LPIPS loss functions',
        'Complete training loop and evaluation metrics',
    ],
    'Notebook 06: MVSplat Code Walkthrough': [
        'Official repository structure and key files',
        'Training config (Hydra), data pipeline (RE10K)',
    ],
    'Notebook 07: Inference & Evaluation': [
        'Inference pipeline and speed benchmarking',
        'Failure case analysis and depth quality',
    ],
    'Notebook 08: DepthSplat & 2025 Advances': [
        'Depth priors (Depth Anything V2), adaptive planes',
        'Single-image 3D, multi-view scaling, 2025 trends',
    ],
    'Notebook 09: MVSplat vs pixelSplat': [
        'Architecture, benchmark, and qualitative comparison',
        'Decision framework for method selection',
    ],
}

for nb, topics in phase4_summary.items():
    print(f"\n{nb}")
    for topic in topics:
        print(f"  \u2713 {topic}")

print("\n" + "="*70)
print("\nKey Takeaways:")
print("1. Feed-forward 3DGS trades quality for speed (seconds vs minutes)")
print("2. MVSplat uses explicit geometry (cost volume) → faster, structured")
print("3. pixelSplat uses implicit geometry (attention) → flexible, slower")
print("4. DepthSplat shows depth priors significantly improve quality")
print("5. The field is rapidly evolving: single-image, N-view, video-based")
print("6. Next: VGGT (Phase 5) for foundation-model-based 3D vision")

In [ ]:
# Final visualization: Phase 4 knowledge map

fig, ax = plt.subplots(1, 1, figsize=(18, 10))
ax.set_xlim(0, 18); ax.set_ylim(0, 11)
ax.axis('off')
ax.set_title('Phase 4 Knowledge Map: Feed-forward 3DGS',
             fontsize=16, fontweight='bold', pad=15)

# Notebook nodes
nodes = [
    (2, 9, 'NB00\nOverview', '#E0E0E0'),
    (5, 9, 'NB01\nCost Volume', '#BBDEFB'),
    (8, 9, 'NB02\nPixel-aligned', '#C8E6C9'),
    (4, 6.5, 'NB03\nMVSplat', '#BBDEFB'),
    (8, 6.5, 'NB04\npixelSplat', '#FFE0B2'),
    (12, 6.5, 'NB05\nTraining', '#E1BEE7'),
    (4, 4, 'NB06\nCode Walk', '#BBDEFB'),
    (8, 4, 'NB07\nInference', '#FFF9C4'),
    (12, 4, 'NB08\nDepthSplat', '#C8E6C9'),
    (8, 1.5, 'NB09\nComparison', '#FFCDD2'),
]

for x, y, text, color in nodes:
    box = FancyBboxPatch((x-0.8, y-0.5), 1.6, 1.0, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='black', lw=1.5)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=8, fontweight='bold')

# Connections (prerequisite flows)
connections = [
    (2, 8.5, 5, 8.5),     # 00 → 01
    (5, 8.5, 8, 8.5),     # 01 → 02
    (5, 8.5, 4, 7.0),     # 01 → 03
    (8, 8.5, 8, 7.0),     # 02 → 04
    (4, 7.0, 4, 4.5),     # 03 → 06
    (4, 7.0, 8, 7.0),     # 03 → 04
    (4, 6.0, 8, 4.5),     # 03 → 07
    (8, 6.0, 8, 4.5),     # 04 → 07
    (12, 6.0, 8, 4.5),    # 05 → 07
    (8, 3.5, 8, 2.0),     # 07 → 09
    (12, 3.5, 12, 4.5),   # 08 ← 05
    (4, 3.5, 8, 2.0),     # 06 → 09
    (12, 3.5, 8, 2.0),    # 08 → 09
]

for x1, y1, x2, y2 in connections:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#999', lw=1, alpha=0.5))

# Legend
legend_items = [
    ('#BBDEFB', 'MVSplat-focused'),
    ('#FFE0B2', 'pixelSplat-focused'),
    ('#C8E6C9', 'Fundamentals / DepthSplat'),
    ('#E1BEE7', 'Training'),
    ('#FFCDD2', 'Comparison'),
]

for i, (color, label) in enumerate(legend_items):
    y = 9.5 - i * 0.5
    rect = plt.Rectangle((14.5, y-0.15), 0.3, 0.3, facecolor=color, edgecolor='black')
    ax.add_patch(rect)
    ax.text(15.0, y, label, fontsize=8, va='center')

# Next phase arrow
ax.annotate('Phase 5: VGGT →', xy=(16, 1.5), fontsize=11, fontweight='bold',
            color='#9C27B0')

plt.tight_layout()
plt.show()

print("\n" + "\u2728" * 20)
print("   Phase 4: Feed-forward Gaussian Splatting - COMPLETE!")
print("\u2728" * 20)

## What's Next?

**Phase 5: VGGT (Visual Geometry Grounded Transformer)** - Foundation models for multi-view 3D vision. Native N-view processing, no pose estimation needed, multi-task prediction.

---

## References

1. MVSplat: https://arxiv.org/abs/2403.14627
2. pixelSplat: https://arxiv.org/abs/2312.12337
3. DepthSplat: https://arxiv.org/abs/2412.18010
4. Depth Anything V2: https://arxiv.org/abs/2406.09414
5. Splatt3R: https://arxiv.org/abs/2408.07648
6. Flash3D: https://arxiv.org/abs/2406.04343
7. VGGT: https://arxiv.org/abs/2503.11651
8. 3D Gaussian Splatting: https://arxiv.org/abs/2308.14737